# 02 · Annotation
**Owner:** _your name_ · Group 20 · Dutch Housing Blame

**Research question.** Who do people blame for the Dutch housing crisis online, and do Dutch-language
(r/thenetherlands) and English-language (r/Netherlands) communities blame different actors?

**Goal of this notebook.** Turn our annotation scheme into a codebook, draw the samples, and measure how well we agree.

**Input:** `data/processed/comments_housing.parquet` (from 01)
**Output:** `annotation/codebook.md`, the sample files, `annotation/labels_final.csv`, agreement table.

**Done when:**
- [ ] codebook agreed by all four
- [ ] samples follow the design (check cells)
- [ ] kappa per pair and pooled
- [ ] `labels_final.csv` follows the contract

**How to work:** each task has a few parts. A part says what has to exist when you're done, not how to build it –
split it into as many cells and functions as you like. Names in `code font` are used by the check cells or by other
notebooks, so keep them. *Go further* items are optional, for when we need more depth.

Read `notebooks/GUIDE.md` first (rules + the column names everyone uses).

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 20
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
ANNOTATION = ROOT / "annotation"
FIGURES = ROOT / "figures"
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

## Task 1 · Codebook
**Why (for the paper):** Clear definitions are the basis of reliable labels; the codebook goes into the method section/appendix.

**1A. Labels**
- `annotation/codebook.md` with every label from our scheme: definition, values, decision rule and a *paraphrased* example (never a real quote).
- What counts for each blame target, including the candidates *banks/mortgage system* and *Airbnb/tourism*.

**1B. Hard cases**
- At least six hard cases with how to label them (sarcasm, quotes, questions, blaming "everyone", a policy instead of an actor, ...).
- A decision on adding *treatment* responsibility ("who should fix it") – read Iyengar, *Is Anyone Responsible?* (1991).

**Go further (optional):**
- Three sentences for the related-work section on how our scheme relates to Iyengar.

## Task 2 · Calibration
**Why (for the paper):** Shows the codebook works before we spend hours on 300 comments.

**2A. Calibration sample**
- `pool`: the comments that are eligible for annotation (decide the rules, e.g. minimum length, language).
- 25 comments from both subreddits exported for all four annotators, with the context they need.

**2B. Compare and agree**
- Where do the four of us disagree – per label, and which comments?
- What changed in the codebook after the discussion, and the frozen target list (write it below).

**Go further (optional):**
- Fleiss' kappa for the four annotators.

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 3 · Main sample
**Why (for the paper):** The sample design decides how far we can generalise; it must be explained and reproducible.

**3A. Design**
- A written decision: only `kw_match` comments, or also thread-only comments? (Many keyword matches are renting questions without blame.)
- `sample`: 300 comments, 150 per subreddit, spread over the quarters, no calibration comments, max 1 per user, max 2 per thread.

**3B. Sets**
- `set_a` and `set_b`: 150 each, 75 per subreddit, shuffled.
- One file per annotator in `annotation/`.

In [ ]:
# check (do not change)
assert len(sample) == 300 and sample["id"].is_unique
assert sample["subreddit"].value_counts().eq(150).all()
assert sample["author_hash"].is_unique
assert sample.groupby("link_id").size().max() <= 2
for s in (set_a, set_b):
    assert len(s) == 150 and s["subreddit"].value_counts().eq(75).all()
assert not set(set_a["id"]) & set(set_b["id"])
print("ok")

**Go further (optional):**
- A table comparing the sample with all housing comments (subreddit, language, quarter).

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 4 · Agreement
**Why (for the paper):** Reliability of the human labels is the first thing reviewers check.

**4A. Kappa**
- `kappa`: a table with `pair`, `category`, `kappa`, `n` for pair A, pair B and pooled.
- Blame targets are multi-label – one yes/no kappa per target.
- A decision on how `unclear` is treated.

**4B. Reading it**
- Plain % agreement next to kappa, and why the two can differ for rare labels.

In [ ]:
# check (do not change)
assert {"pair", "category", "kappa", "n"}.issubset(kappa.columns)
assert kappa["kappa"].between(-1, 1).all()
print(kappa.round(2))

**Go further (optional):**
- Confusion matrix for the weakest category.
- Interpretation following Landis & Koch (1977).

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 5 · Final labels
**Why (for the paper):** One clean label file is what notebooks 03 and 04 build on.

**5A. Adjudication**
- Every disagreement resolved, with a record of how.

**5B. Label file**
- `annotation/labels_final.csv` exactly like the contract in the guide.
- A `split` column: about 100 `heldout` comments (stratified by subreddit), the rest `main`.

In [ ]:
# check (do not change)
labels = pd.read_csv(ANNOTATION / "labels_final.csv")
assert labels["id"].is_unique and len(labels) == 300
assert set(labels["split"]) == {"main", "heldout"}
assert any(c.startswith("target_") for c in labels.columns)
print("ok")

**Go further (optional):**
- Label frequencies per subreddit (a preview for notebook 03).

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Notes for the paper
- Sample design and why.
- Kappa table and what changed after calibration.
- Codebook in the appendix.